In [12]:
import numpy as np
import shapely
import trimesh
import triangle # Assuming _constrained_triangulate uses this
from typing import Dict, List, Tuple

# Make sure _constrained_triangulate is defined as before:
def _constrained_triangulate(polygon:shapely.Geometry):
    """Triangulate a polygon with holes using 'triangle' library."""
    # Check if triangle is installed
    try:
        import triangle
    except ImportError:
        print("Error: 'triangle' library is required. Install with 'pip install triangle'")
        return None, None

    # Ensure polygon is valid before proceeding
    if not polygon.is_valid:
        polygon = polygon.buffer(0)
        if not polygon.is_valid:
             print("Warning: Could not validate polygon for triangulation. Skipping.")
             return None, None
    if polygon.is_empty or polygon.area == 0:
        print("Warning: Skipping empty polygon for triangulation.")
        return None, None

    ext = np.array(polygon.exterior.coords[:-1])  # Outer boundary (N, 2)
    holes_coords = [np.array(hole.coords[:-1]) for hole in polygon.interiors]  # Inner boundaries

    # --- Prepare input for triangle ---
    vertices = ext # Start with exterior vertices
    segments = np.column_stack([np.arange(len(ext)), np.roll(np.arange(len(ext)), -1)]) # Exterior segments

    holes_points = [] # Points inside holes for triangle library
    start_index = len(vertices)
    for hole_coords in holes_coords:
        if len(hole_coords) < 3: # Skip degenerate holes
            print("Warning: Skipping degenerate hole in triangulation.")
            continue
        # Add hole vertices
        vertices = np.vstack([vertices, hole_coords])
        # Add hole segments
        hole_segments = np.column_stack([
            np.arange(len(hole_coords)),
            np.roll(np.arange(len(hole_coords)), -1)
        ]) + start_index
        segments = np.vstack([segments, hole_segments])
        # Calculate a point inside the hole
        try:
            hole_poly = shapely.Polygon(hole_coords)
            # Ensure representative_point is within the hole, otherwise use centroid
            rep_point = hole_poly.representative_point()
            if not hole_poly.contains(rep_point):
                 rep_point = hole_poly.centroid # Fallback to centroid
            holes_points.append(rep_point.coords[0])
        except Exception as e:
            print(f"Warning: Could not process hole for triangulation: {e}. Skipping hole.")
            # If hole processing fails, we might need to remove its vertices/segments
            # For simplicity, we continue, but this could lead to issues.
            vertices = vertices[:-len(hole_coords)]
            segments = segments[:-len(hole_coords)]
            continue # Skip adding this hole's representative point

        start_index += len(hole_coords)

    tri_input = {'vertices': vertices, 'segments': segments}
    if holes_points:
        tri_input['holes'] = holes_points

    # --- Perform constrained Delaunay triangulation ---
    try:
        # 'p' ensures polygon boundaries are respected (CDT)
        # 'q' requests quality mesh generation (optional, adds Steiner points)
        # 'a' imposes max triangle area (optional)
        # Using 'p' only for pure CDT without adding points inside
        tri = triangle.triangulate(tri_input, 'p')

        if 'vertices' not in tri or 'triangles' not in tri or len(tri['vertices'])==0 or len(tri['triangles'])==0:
             print(f"Warning: Triangulation resulted in empty output. Skipping polygon.")
             return None, None

        # Important: Ensure vertices returned by triangle match the input order for boundaries
        # The 'vertices' output by triangle might be reordered or include Steiner points if 'q' is used.
        # For 'p' only, the original vertices should be preserved but potentially reordered.
        # We need the original boundary vertices in order for skinning.
        # Return the original exterior points separately for skinning.
        return np.array(tri['vertices']), np.array(tri['triangles']), ext # Return boundary points too

    except Exception as e:
        print(f"Error during triangulation: {e}")
        return None, None, None


def make_mesh(polydict: Dict[int, shapely.Geometry], downsample: int, section_spacing_um: float = 20.0):
    """
    Creates a 3D mesh by 'skinning' contours between sections.
    Includes triangulated caps and side walls connecting adjacent contours.
    NOTE: Side wall generation uses a simplified sequential connection strategy.
    """
    all_vertices_list = []
    cap_faces_list = []
    side_faces_list = []
    vertex_offset = 0
    scale = 1 / (2**downsample) # um per pixel

    sorted_sections = sorted(polydict.keys())
    if not sorted_sections:
        print("Warning: polydict is empty. Cannot create mesh.")
        return None

    section_boundaries_3d = {} # Store 3D boundary points {secnum: boundary_verts_3d}
    section_boundary_global_indices = {} # Store global indices {secnum: [indices]}

    # --- Pass 1: Collect all boundary vertices globally ---
    print("Pass 1: Collecting boundary vertices...")
    for secnum in sorted_sections:
        poly = polydict[secnum]
        z = secnum * section_spacing_um * scale # Z position in um

        # --- Process Polygon/MultiPolygon ---
        poly_arr = []
        if poly.geom_type == 'Polygon':
            if poly.is_valid and poly.area > 0: poly_arr.append(poly)
            elif poly.buffer(0).is_valid and poly.buffer(0).area > 0: poly_arr.append(poly.buffer(0))
        elif poly.geom_type == 'MultiPolygon':
            valid_polys = [p.buffer(0) for p in poly.geoms if p.is_valid or p.buffer(0).is_valid]
            poly_arr = [p for p in valid_polys if p.area > 0]

        if not poly_arr:
            print(f"Warning: No valid geometry found for section {secnum}. Skipping.")
            section_boundaries_3d[secnum] = np.empty((0, 3))
            section_boundary_global_indices[secnum] = []
            continue

        # For simplicity, combine boundaries if MultiPolygon. Refine if needed.
        combined_boundary_2d = []
        for poly_i in poly_arr:
             if len(poly_i.exterior.coords) > 1:
                 combined_boundary_2d.append(np.array(poly_i.exterior.coords[:-1])) # Exclude duplicate closing point

        if not combined_boundary_2d:
             section_boundaries_3d[secnum] = np.empty((0, 3))
             section_boundary_global_indices[secnum] = []
             continue

        section_boundary_2d = np.vstack(combined_boundary_2d)
        section_boundary_3d = np.hstack([(section_boundary_2d * scale), np.full((section_boundary_2d.shape[0], 1), z)])

        num_boundary_verts = len(section_boundary_3d)
        section_boundaries_3d[secnum] = section_boundary_3d
        section_boundary_global_indices[secnum] = list(range(vertex_offset, vertex_offset + num_boundary_verts))

        all_vertices_list.append(section_boundary_3d)
        vertex_offset += num_boundary_verts

    # --- Pass 2: Triangulate caps and collect internal vertices ---
    print("Pass 2: Triangulating caps and collecting internal vertices...")
    internal_vertices_start_offset = vertex_offset # Start index for non-boundary vertices
    for idx, secnum in enumerate(sorted_sections):
        # Only triangulate fully for the first and last sections (caps)
        is_cap_section = (idx == 0 or idx == len(sorted_sections) - 1)

        poly = polydict[secnum] # Get original polygon again

        # --- Process Polygon/MultiPolygon (similar to Pass 1) ---
        poly_arr = []
        if poly.geom_type == 'Polygon':
            if poly.is_valid and poly.area > 0: poly_arr.append(poly)
            elif poly.buffer(0).is_valid and poly.buffer(0).area > 0: poly_arr.append(poly.buffer(0))
        elif poly.geom_type == 'MultiPolygon':
            valid_polys = [p.buffer(0) for p in poly.geoms if p.is_valid or p.buffer(0).is_valid]
            poly_arr = [p for p in valid_polys if p.area > 0]

        if not poly_arr: continue # Skip if no valid geometry

        z = secnum * section_spacing_um * scale # Z position in um

        for poly_i in poly_arr:
            try:
                # Triangulate the polygon (including holes)
                # We need the mapping from local tri_verts indices to global indices
                tri_verts_2d, faces_local, boundary_verts_2d = _constrained_triangulate(poly_i)

                if tri_verts_2d is None or faces_local is None:
                    print(f"Warning: Triangulation failed for polygon in section {secnum}. Skipping.")
                    continue

                tri_verts_3d = np.hstack([(tri_verts_2d * scale), np.full((tri_verts_2d.shape[0], 1), z)])

                # --- Map local triangulation vertices to global indices ---
                # Boundary vertices are already in the global list. Find their indices.
                # Internal vertices need to be added.
                global_indices_map = {} # Map local index -> global index
                internal_vertices_to_add = []
                local_internal_indices = []

                # Find global indices for boundary vertices (use stored indices)
                boundary_global_indices = section_boundary_global_indices[secnum]
                num_boundary_verts = len(boundary_global_indices)

                # Assuming the first N vertices from triangulation correspond to the boundary
                # This relies heavily on _constrained_triangulate preserving boundary vertex order at the start
                if tri_verts_2d.shape[0] >= num_boundary_verts:
                    for i in range(num_boundary_verts):
                        global_indices_map[i] = boundary_global_indices[i]
                else:
                    print(f"Warning: Mismatch between boundary count ({num_boundary_verts}) and triangulation vertices ({tri_verts_2d.shape[0]}) in section {secnum}. Cap faces might be incorrect.")
                    # Attempt partial mapping? Or skip cap? For now, continue with potentially wrong mapping.
                    for i in range(min(num_boundary_verts, tri_verts_2d.shape[0])):
                         global_indices_map[i] = boundary_global_indices[i]


                # Identify and collect internal vertices
                for i in range(num_boundary_verts, tri_verts_2d.shape[0]):
                    local_internal_indices.append(i)
                    internal_vertices_to_add.append(tri_verts_3d[i])
                    # Assign new global indices for internal vertices
                    global_indices_map[i] = vertex_offset # Use current offset before incrementing
                    vertex_offset += 1

                # Add internal vertices to the global list
                if internal_vertices_to_add:
                    all_vertices_list.append(np.vstack(internal_vertices_to_add))

                # Remap local faces to global indices if it's a cap section
                if is_cap_section:
                    faces_global = []
                    for face in faces_local:
                        try:
                            global_face = (global_indices_map[face[0]],
                                           global_indices_map[face[1]],
                                           global_indices_map[face[2]])
                            faces_global.append(global_face)
                        except KeyError:
                            print(f"Warning: Could not map local face vertex index in section {secnum}. Skipping face.")
                            continue # Skip face if mapping fails
                    if faces_global:
                        cap_faces_list.append(np.array(faces_global, dtype=int))

            except Exception as e:
                 print(f"Error processing polygon triangulation in section {secnum}: {e}")
                 continue

    # --- Pass 3: Create side walls ---
    print("Pass 3: Creating side walls...")
    for i in range(len(sorted_sections) - 1):
        sec_i = sorted_sections[i]
        sec_j = sorted_sections[i+1]

        # Get the global indices for the boundaries of adjacent sections
        global_indices_i = section_boundary_global_indices.get(sec_i, [])
        global_indices_j = section_boundary_global_indices.get(sec_j, [])

        num_boundary_i = len(global_indices_i)
        num_boundary_j = len(global_indices_j)

        if num_boundary_i < 2 or num_boundary_j < 2: # Need at least 2 points per loop
            continue

        # --- Simplified Sequential Connection ---
        # Iterate based on the shorter loop length
        min_len = min(num_boundary_i, num_boundary_j)
        for k in range(min_len):
            # Get current and next global indices for both loops, handling wrap-around
            g_idx1_curr = global_indices_i[k]
            g_idx1_next = global_indices_i[(k + 1) % num_boundary_i]
            g_idx2_curr = global_indices_j[k]
            g_idx2_next = global_indices_j[(k + 1) % num_boundary_j]

            # Create two triangles to form a quad connecting the edges
            side_faces_list.append((g_idx1_curr, g_idx2_curr, g_idx2_next))
            side_faces_list.append((g_idx1_curr, g_idx2_next, g_idx1_next))

        # --- Rudimentary Handling for Mismatched Lengths ---
        # Connect remaining vertices on the longer loop to the last matched vertex on the shorter loop
        if num_boundary_i > min_len:
            g_idx2_last = global_indices_j[(min_len - 1)] # Last matched index on loop j
            for k in range(min_len, num_boundary_i):
                g_idx1_curr = global_indices_i[k]
                g_idx1_next = global_indices_i[(k + 1) % num_boundary_i]
                side_faces_list.append((g_idx1_curr, g_idx2_last, g_idx1_next)) # Triangle fan
        elif num_boundary_j > min_len:
            g_idx1_last = global_indices_i[(min_len - 1)] # Last matched index on loop i
            for k in range(min_len, num_boundary_j):
                g_idx2_curr = global_indices_j[k]
                g_idx2_next = global_indices_j[(k + 1) % num_boundary_j]
                side_faces_list.append((g_idx1_last, g_idx2_curr, g_idx2_next)) # Triangle fan


    # --- Final Assembly ---
    print("Final Assembly...")
    if not all_vertices_list:
        print("Error: No vertices collected.")
        return None

    all_vertices = np.vstack(all_vertices_list)

    all_faces_list = []
    if cap_faces_list:
        all_faces_list.append(np.vstack(cap_faces_list))
    if side_faces_list:
        all_faces_list.append(np.array(side_faces_list, dtype=int))

    if not all_faces_list:
        print("Error: No faces (caps or sides) generated.")
        # Return a point cloud maybe? Or None.
        # return trimesh.PointCloud(all_vertices)
        return None

    all_faces = np.vstack(all_faces_list)

    if all_vertices.shape[0] == 0 or all_faces.shape[0] == 0:
        print("Error: Mesh generation resulted in empty vertices or faces.")
        return None

    # --- Create Trimesh object ---
    try:
        print(f"Creating Trimesh object with {all_vertices.shape[0]} vertices and {all_faces.shape[0]} faces...")
        mesh = trimesh.Trimesh(vertices=all_vertices, faces=all_faces, process=False) # Process later if needed

        # Validate vertex indices in faces
        max_vertex_index = all_vertices.shape[0] - 1
        if np.any(all_faces > max_vertex_index):
             print(f"Error: Face indices ({np.max(all_faces)}) exceed vertex count ({max_vertex_index}).")
             # Find problematic faces
             bad_faces = all_faces[np.any(all_faces > max_vertex_index, axis=1)]
             print("Problematic faces (first 5):", bad_faces[:5])
             # np.savez("debug_mesh_data_bad_indices.npz", vertices=all_vertices, faces=all_faces)
             return None # Cannot create valid mesh

        # Optional: Clean up the mesh
        print("Processing mesh (merging vertices, removing duplicates)...")
        mesh.process() # Merges duplicate vertices, removes duplicate faces etc.

        # Further checks (can be slow):
        # if not mesh.is_watertight:
        #     print("Warning: Generated mesh is not watertight.")
        # if not mesh.is_manifold:
        #     print("Warning: Generated mesh is not manifold.")

        print("Mesh generation complete.")
        return mesh
    except Exception as e:
        print(f"Error creating final Trimesh object: {e}")
        print(f"Vertices shape: {all_vertices.shape}, Faces shape: {all_faces.shape}")
        # np.savez("debug_mesh_data_error.npz", vertices=all_vertices, faces=all_faces)
        return None



In [2]:
import sys
sys.path.append('..')

from ontology_handling import TreeHelper
from dharani_functions import DharaniHelper


In [3]:
helper = DharaniHelper(specimennum=2)

In [4]:
%%time 
annotations_by_ontoid = helper.get_annotations(concurrent=True)

CPU times: total: 1.55 s
Wall time: 1min 39s


In [6]:
dharani_onto_helper = TreeHelper('dharani')

Loading ontology data for 'dharani' from source...
Finished loading ontology data for 'dharani'.


In [7]:
cc_successors = dharani_onto_helper.get_successor_ids(238)

In [8]:
from collections import defaultdict

cc_annotations = defaultdict(dict)

for ch in [238]+cc_successors:
    if ch in annotations_by_ontoid:
        for secno in annotations_by_ontoid[ch]:
            cc_annotations[secno][ch]=annotations_by_ontoid[ch][secno]

In [9]:
shape_dict = {}

for secno in sorted(cc_annotations):
    print(secno,cc_annotations[secno].keys())
    
    united = None
    for ontoid in cc_annotations[secno]:
        shp = cc_annotations[secno][ontoid]
        if united is None:
            united = shp
        else:
            united = united.union(shp)
    
    shape_dict[secno]=united

598 dict_keys([238])
619 dict_keys([238])
625 dict_keys([238])
634 dict_keys([238])
652 dict_keys([238])
667 dict_keys([238])
682 dict_keys([238])
694 dict_keys([238])
709 dict_keys([238])
718 dict_keys([238])
730 dict_keys([238])
742 dict_keys([238])
757 dict_keys([238])
772 dict_keys([238])
784 dict_keys([238])
790 dict_keys([238])
916 dict_keys([238])
934 dict_keys([238])
946 dict_keys([238])
964 dict_keys([238])
976 dict_keys([238])
1000 dict_keys([238])
1015 dict_keys([238])


In [13]:
cc_mesh = make_mesh(shape_dict, helper._downsample)

Pass 1: Collecting boundary vertices...
Pass 2: Triangulating caps and collecting internal vertices...
Pass 3: Creating side walls...
Final Assembly...
Creating Trimesh object with 5892 vertices and 11780 faces...
Processing mesh (merging vertices, removing duplicates)...
Mesh generation complete.


In [14]:
cc_mesh.show()